In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
LAYERS = [
    "conv1",
    "layer1.0", "layer1.1", "layer1.2",
    "layer2.0", "layer2.1", "layer2.2", "layer2.3",
    "layer3.0", "layer3.1", "layer3.2", "layer3.3", "layer3.4", "layer3.5",
    "layer4.0", "layer4.1", "layer4.2",
    "avgpool",
]

EMERGE_METRIC    = "balanced"     # "balanced" | "plain"
EMERGE_FRAC      = 0.90
EMERGE_CRITERION = "sharp_jump"   # "frac90" | "sharp_jump"

emerge_col = "emerge_idx_frac"   if EMERGE_CRITERION == "frac90" else "emerge_idx_jump"
layer_col  = "emerge_layer_frac" if EMERGE_CRITERION == "frac90" else "emerge_layer_jump"

GAMMAS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]

print(f"EMERGE_METRIC    : {EMERGE_METRIC}")
print(f"EMERGE_FRAC      : {EMERGE_FRAC}")
print(f"EMERGE_CRITERION : {EMERGE_CRITERION}")
print(f"GAMMAS           : {GAMMAS}")

EMERGE_METRIC    : balanced
EMERGE_FRAC      : 0.9
EMERGE_CRITERION : sharp_jump
GAMMAS           : [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]


In [12]:
import sys
ROOT = Path('/scratch/network/cr7998/cv_emergence_project')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FB            = ROOT / 'data' / 'FunnyBirds'
MCBM_FB_FEATS = {g: ROOT / 'features' / f'resnet50_mcbm_funnybirds_gamma{g}' for g in GAMMAS}

assert FB.exists(),                          f'Missing FunnyBirds folder: {FB}'
assert (FB / 'dataset_train.json').exists(), f'Missing dataset_train.json: {FB}'

# Check the actual labels file, not just the directory
def _feats_ready(feat_dir: Path) -> bool:
    return (feat_dir / "labels_train.pt").exists() and (feat_dir / "labels_test.pt").exists()

_missing = [g for g in GAMMAS if not _feats_ready(MCBM_FB_FEATS[g])]
if _missing:
    print(f"WARNING: Features not yet extracted for gammas: {_missing}")
    print("Run your FunnyBirds MCBM feature extraction script for those gammas.")
    print("Example: sbatch run_mcbm_funnybirds_extract.sh")
    GAMMAS = [g for g in GAMMAS if g not in _missing]

if not GAMMAS:
    raise RuntimeError(
        "No MCBM FunnyBirds features are available.\n"
        "Extract features first, then re-run from this cell.\n"
        f"Expected path: {ROOT / 'features' / 'resnet50_mcbm_funnybirds_gamma0.0'}"
    )

for g in GAMMAS:
    print(f"  [ok]   gamma={g}: {MCBM_FB_FEATS[g]}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n[config] device : {device}')
print(f'[config] GAMMAS : {GAMMAS}')

RuntimeError: No MCBM FunnyBirds features are available.
Extract features first, then re-run from this cell.
Expected path: /scratch/network/cr7998/cv_emergence_project/features/resnet50_mcbm_funnybirds_gamma0.0

In [4]:
def load_species_maps(fb_root: Path):
    classes_csv = fb_root / "metadata" / "classes.csv"
    if not classes_csv.exists():
        raise FileNotFoundError(
            "metadata/classes.csv not found. Run prepare_funnybirds_metadata.py first."
        )
    df = pd.read_csv(classes_csv)
    id2name  = dict(zip(df["class_id"], df["class_name"]))
    id2short = {k: v.replace("funnybird_", "FB") for k, v in id2name.items()}
    return id2name, id2short


def load_meta(fb_root: Path) -> pd.DataFrame:
    images_csv = fb_root / "metadata" / "images.csv"
    if not images_csv.exists():
        raise FileNotFoundError(
            "metadata/images.csv not found. Run prepare_funnybirds_metadata.py first."
        )
    df = pd.read_csv(images_csv)
    id2name, _ = load_species_maps(fb_root)
    df["species_id"]   = df["class_id"]
    df["species_name"] = df["class_id"].map(id2name)
    return df


def load_image_attr_labels_robust(fb_root: Path) -> pd.DataFrame:
    concepts_csv = fb_root / "metadata" / "image_concepts_binary.csv"
    if not concepts_csv.exists():
        raise FileNotFoundError(
            "metadata/image_concepts_binary.csv not found. "
            "Run prepare_funnybirds_metadata.py first."
        )
    wide = pd.read_csv(concepts_csv)
    concept_cols = [c for c in wide.columns if c != "image_id"]
    long = wide.melt(
        id_vars="image_id", value_vars=concept_cols,
        var_name="attr_name", value_name="is_present",
    )
    long["attr_id"]    = long.groupby("attr_name", sort=False).ngroup()
    long["is_present"] = long["is_present"].astype(int)
    long["certainty"]  = 1   # FunnyBirds: ground-truth, always certain
    return long[["image_id", "attr_id", "attr_name", "is_present", "certainty"]]


def load_attr_maps(fb_root: Path):
    concepts_csv = fb_root / "metadata" / "concepts.csv"
    if not concepts_csv.exists():
        raise FileNotFoundError(
            "metadata/concepts.csv not found. Run prepare_funnybirds_metadata.py first."
        )
    df = pd.read_csv(concepts_csv)
    id2name = dict(zip(df["concept_id"], df["concept_name"]))
    name2id = dict(zip(df["concept_name"], df["concept_id"]))
    return id2name, name2id


print("Defined: load_species_maps  load_meta  load_image_attr_labels_robust  load_attr_maps")

Defined: load_species_maps  load_meta  load_image_attr_labels_robust  load_attr_maps


In [5]:
from datasets.funnybirds_dataset import concept_names as _fb_concept_names

ATTR_LIST = _fb_concept_names()
print(f"FunnyBirds concepts ({len(ATTR_LIST)}): {ATTR_LIST}")

FunnyBirds concepts (26): ['beak_0', 'beak_1', 'beak_2', 'beak_3', 'eye_0', 'eye_1', 'eye_2', 'wing_0', 'wing_1', 'wing_2', 'wing_3', 'wing_4', 'wing_5', 'foot_0', 'foot_1', 'foot_2', 'foot_3', 'tail_0', 'tail_1', 'tail_2', 'tail_3', 'tail_4', 'tail_5', 'tail_6', 'tail_7', 'tail_8']


In [6]:
from datasets.funnybirds_dataset import FunnyBirdsDataset

_fb_ds = FunnyBirdsDataset(FB, split="train")
class_concept_matrix, _ = _fb_ds.get_class_concept_matrix()

cc_df = pd.DataFrame(
    class_concept_matrix.numpy(),
    columns=_fb_concept_names(),
    index=[f"funnybird_{i:02d}" for i in range(class_concept_matrix.shape[0])],
)
print(f"Class-concept matrix shape: {cc_df.shape}")
print("Each row should sum to 5 (one variant per part per species):")
print(cc_df.sum(axis=1).value_counts())
print("\nPart groups (columns starting with each prefix):")
for part in ["beak", "wing", "tail", "foot", "eye"]:
    print(f"  {part}: {[c for c in cc_df.columns if c.startswith(part)]}")
cc_df.head()

Class-concept matrix shape: (50, 26)
Each row should sum to 5 (one variant per part per species):
5.0    30
3.0     7
4.0     4
2.0     4
0.0     3
1.0     2
Name: count, dtype: int64

Part groups (columns starting with each prefix):
  beak: ['beak_0', 'beak_1', 'beak_2', 'beak_3']
  wing: ['wing_0', 'wing_1', 'wing_2', 'wing_3', 'wing_4', 'wing_5']
  tail: ['tail_0', 'tail_1', 'tail_2', 'tail_3', 'tail_4', 'tail_5', 'tail_6', 'tail_7', 'tail_8']
  foot: ['foot_0', 'foot_1', 'foot_2', 'foot_3']
  eye: ['eye_0', 'eye_1', 'eye_2']


,beak_0,beak_1,beak_2,beak_3,eye_0,eye_1,eye_2,wing_0,wing_1,wing_2,...,foot_3,tail_0,tail_1,tail_2,tail_3,tail_4,tail_5,tail_6,tail_7,tail_8
funnybird_00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
funnybird_01,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
funnybird_02,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
funnybird_03,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
funnybird_04,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
meta          = load_meta(FB)
img_attr_long = load_image_attr_labels_robust(FB)
attr_id_to_name, attr_name_to_id = load_attr_maps(FB)

attr_df = pd.DataFrame({
    "attr_name": list(attr_name_to_id.keys()),
    "attr_id":   list(attr_name_to_id.values()),
})

id2name, _ = load_species_maps(FB)
def spname(sid: int) -> str:
    return id2name.get(int(sid), f"funnybird_{int(sid):02d}")

print(f"meta          : {len(meta)} images  "
      f"({meta['is_train'].sum()} train, {(meta['is_train']==0).sum()} test)")
print(f"img_attr_long : {len(img_attr_long)} rows, "
      f"{img_attr_long['attr_name'].nunique()} concepts")
print(f"attr_df       : {len(attr_df)} concepts")
print(f"Concepts: {list(attr_name_to_id.keys())}")

meta          : 50500 images  (50000 train, 500 test)
img_attr_long : 1313000 rows, 26 concepts
attr_df       : 26 concepts
Concepts: ['beak_0', 'beak_1', 'beak_2', 'beak_3', 'eye_0', 'eye_1', 'eye_2', 'wing_0', 'wing_1', 'wing_2', 'wing_3', 'wing_4', 'wing_5', 'foot_0', 'foot_1', 'foot_2', 'foot_3', 'tail_0', 'tail_1', 'tail_2', 'tail_3', 'tail_4', 'tail_5', 'tail_6', 'tail_7', 'tail_8']


In [8]:
def safe_torch_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")


def load_features(feat_dir: Path, layer: str, split: str) -> torch.Tensor:
    p = feat_dir / f"{layer}_{split}.pt"
    assert p.exists(), f"Missing: {p}"
    X = safe_torch_load(p)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    return X.float()


def to_1d_int_array(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    return np.array(x).reshape(-1).astype(int)


def infer_kind(arr):
    if arr.max() <= 200 and arr.min() >= 0:
        return "species_id_like"
    if arr.max() > 200:
        return "image_id_like"
    return "unknown"


def load_split_order(feat_dir, split):
    p = feat_dir / f"labels_{split}.pt"
    assert p.exists(), f"Missing: {p}"
    t = safe_torch_load(p)
    assert isinstance(t, dict), f"Expected dict in {p}, got {type(t)}"
    assert "image_ids" in t, f"{p} missing 'image_ids' key; has {list(t.keys())}"
    ids  = to_1d_int_array(t["image_ids"])
    kind = infer_kind(ids)
    return kind, ids


print("Defined: safe_torch_load  load_features  load_split_order")

Defined: safe_torch_load  load_features  load_split_order


In [9]:
_ref_feat = MCBM_FB_FEATS[0.0]
mcbm_kind_tr, mcbm_ids_tr = load_split_order(_ref_feat, "train")
mcbm_kind_te, mcbm_ids_te = load_split_order(_ref_feat, "test")

print(f"MCBM FB split order: {mcbm_kind_tr}  id range: "
      f"({mcbm_ids_tr.min()}, {mcbm_ids_tr.max()})")
assert mcbm_kind_tr == "image_id_like", "Expected image_id_like for alignment to work"
assert mcbm_kind_te == "image_id_like", "Expected image_id_like for alignment to work"

_ref_ids_tr = np.asarray(mcbm_ids_tr, dtype=int)
_ref_ids_te = np.asarray(mcbm_ids_te, dtype=int)

meta_tr_set = set(meta[meta["is_train"] == 1]["image_id"].astype(int))
meta_te_set = set(meta[meta["is_train"] == 0]["image_id"].astype(int))

keep_tr_idx = np.array([i for i, iid in enumerate(_ref_ids_tr) if iid in meta_tr_set], dtype=int)
keep_te_idx = np.array([i for i, iid in enumerate(_ref_ids_te) if iid in meta_te_set], dtype=int)

aligned_tr_ids = _ref_ids_tr[keep_tr_idx]
aligned_te_ids = _ref_ids_te[keep_te_idx]

_meta_idx = meta.set_index("image_id")
_sid_min  = int(meta["species_id"].min())

ysp_tr = _meta_idx.loc[aligned_tr_ids, "species_id"].to_numpy(dtype=np.int64) - _sid_min
ysp_te = _meta_idx.loc[aligned_te_ids, "species_id"].to_numpy(dtype=np.int64) - _sid_min

assert ysp_tr.min() >= 0, "Negative class index — species_id offset wrong!"
print(f"Aligned: train={len(aligned_tr_ids)}, test={len(aligned_te_ids)}")
print(f"species_id range: {_sid_min} → {int(meta['species_id'].max())}  (offset={_sid_min})")
print(f"ysp_tr range: {ysp_tr.min()} → {ysp_tr.max()}")

AssertionError: Missing: /scratch/network/cr7998/cv_emergence_project/features/resnet50_mcbm_funnybirds_gamma0.0/labels_train.pt

In [10]:
def balanced_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.astype(int).reshape(-1)
    y_pred = y_pred.astype(int).reshape(-1)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return 0.5 * (tpr + tnr)


def frac_of_final_idx(vals: np.ndarray, frac: float = None) -> int:
    if frac is None:
        frac = EMERGE_FRAC
    v = pd.Series(np.asarray(vals, dtype=float)).ffill().bfill().to_numpy()
    target = frac * float(v[-1])
    for i, x in enumerate(v):
        if float(x) >= target:
            return int(i)
    return int(len(v) - 1)


def sharp_rise_idx(vals: np.ndarray) -> int:
    v = pd.Series(np.asarray(vals, dtype=float)).ffill().bfill().to_numpy()
    return int(np.argmax(np.diff(v))) + 1


def train_linear_probe_multiclass(
    Xtr, ytr, Xte, yte, *,
    epochs=6, lr=3e-3, wd=1e-4, seed=0, metric="plain",
):
    torch.manual_seed(seed)
    Xtr_t = torch.as_tensor(Xtr, dtype=torch.float32, device=device)
    ytr_t = torch.as_tensor(ytr, dtype=torch.long,    device=device)
    Xte_t = torch.as_tensor(Xte, dtype=torch.float32, device=device)
    d, C  = Xtr_t.shape[1], int(ytr_t.max().item()) + 1
    model = nn.Linear(d, C).to(device)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xtr_t), ytr_t).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(Xte_t).argmax(dim=1).cpu().numpy()
    if metric == "balanced":
        classes   = np.unique(yte)
        per_class = [float((pred[yte == c] == c).mean()) for c in classes if (yte == c).sum() > 0]
        return float(np.mean(per_class)) if per_class else 0.0
    return float((pred == yte).mean())


def train_linear_probe_binary_weighted(
    Xtr, ytr, Xte, yte, *,
    epochs=8, lr=3e-3, wd=1e-4, seed=0, threshold=0.5,
):
    torch.manual_seed(seed)
    ytr_np = np.asarray(ytr, dtype=np.int32).reshape(-1)
    yte_np = np.asarray(yte, dtype=np.int32).reshape(-1)
    Xtr_t  = torch.as_tensor(Xtr, dtype=torch.float32).to(device)
    Xte_t  = torch.as_tensor(Xte, dtype=torch.float32).to(device)
    ytr_t  = torch.as_tensor(ytr_np, dtype=torch.float32).view(-1, 1).to(device)
    d = int(Xtr_t.shape[1])
    model = nn.Linear(d, 1).to(device)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    pos = float(ytr_np.sum()); neg = float(len(ytr_np) - ytr_np.sum())
    if pos <= 0:
        probs = np.zeros_like(yte_np, dtype=float)
        return 0.5, float((np.zeros_like(yte_np) == yte_np).mean()), 0.0, probs
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(device)
    loss_fn    = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xtr_t), ytr_t).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte_t)).view(-1).cpu().numpy()
    pred = (probs >= threshold).astype(int)
    ba   = balanced_accuracy(yte_np, pred)
    return float(ba), float((pred == yte_np).mean()), float(pred.mean()), probs


print("Defined: balanced_accuracy  frac_of_final_idx  sharp_rise_idx")
print("Defined: train_linear_probe_multiclass  train_linear_probe_binary_weighted")

Defined: balanced_accuracy  frac_of_final_idx  sharp_rise_idx
Defined: train_linear_probe_multiclass  train_linear_probe_binary_weighted


In [11]:
import gc

LAYER = None  # set dynamically below

species_curve = []
for L in LAYERS:
    Xtr_raw = load_features(_ref_feat, L, "train")
    Xte_raw = load_features(_ref_feat, L, "test")
    Xtr_L   = Xtr_raw[keep_tr_idx]; del Xtr_raw
    Xte_L   = Xte_raw[keep_te_idx]; del Xte_raw
    acc = train_linear_probe_multiclass(
        Xtr_L, ysp_tr, Xte_L, ysp_te,
        epochs=15, seed=0, metric=EMERGE_METRIC,
    )
    del Xtr_L, Xte_L
    gc.collect()
    species_curve.append(acc)
    print(f"  {L:12s}  {acc:.4f}")

species_curve = np.asarray(species_curve, dtype=float)

species_emerge_idx_jump = sharp_rise_idx(species_curve)
species_emerge_idx_frac = frac_of_final_idx(species_curve, frac=EMERGE_FRAC)

species_emerge_idx = species_emerge_idx_frac if EMERGE_CRITERION == "frac90" else species_emerge_idx_jump
LAYER = LAYERS[species_emerge_idx]

print(f"\nSpecies emergence:")
print(f"  sharp_jump : {LAYERS[species_emerge_idx_jump]}  "
      f"(acc={species_curve[species_emerge_idx_jump]:.3f})")
print(f"  frac{int(EMERGE_FRAC*100)}     : {LAYERS[species_emerge_idx_frac]}  "
      f"(acc={species_curve[species_emerge_idx_frac]:.3f})")
print(f"\nLAYER set to: {LAYER}  (criterion: {EMERGE_CRITERION})")

AssertionError: Missing: /scratch/network/cr7998/cv_emergence_project/features/resnet50_mcbm_funnybirds_gamma0.0/conv1_train.pt